# 01 — Revival Strategy Mapping

**Input:** `Sehore_WaterBody_Health_Clusters.csv` (from `00_health_score_clustering.ipynb`)

**Goal:** turn each water body's health category into specific, actionable revival
recommendations — this is what actually makes the output a "health card" rather than
just a labeled cluster.

**Uses the six revival strategies already defined in the project report:**

| Code | Strategy |
|---|---|
| P1 | Desilting of Shrinking Water Bodies |
| P2 | Catchment Area Treatment |
| P3 | Rainwater Harvesting Structures |
| P4 | Afforestation Around Water Bodies |
| P5 | Community-Based Monitoring |
| P6 | Sustainable Agricultural Practices |

**Important design decision:** strategy assignment is NOT purely cluster-level. Each
water body also gets checked against individual override rules, because cluster
averages can hide real problems. (Concrete example found in this data: 8 water bodies
sit inside the "Established & Improving" cluster — whose *average* trend is positive —
but are individually still losing water on net. A cluster-only assignment would have
missed all 8 of them.)


In [1]:
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

df = pd.read_csv('../Sehore_WaterBody_Health_Clusters.csv')
print(df.shape)
df.head()


(92, 9)


,water_body_id,area_ha_2024,water_pct_change_capped,catchment_ndvi_change,catchment_pct_cropland,catchment_pct_urban,is_new_since_2017,cluster,health_category
0,00000000000000000000,70.149820,329.366129,-0.027962,46.038576,0.268618,False,1,Established & Improving
1,00000000000000000001,23.641908,500.000000,-0.028710,36.414329,0.399585,True,2,New but Catchment-Stressed
2,00000000000000000002,5.649776,1.923693,-0.044931,81.046048,5.311553,False,1,Established & Improving
3,00000000000000000003,28.894123,4.356884,-0.018878,84.091797,7.364132,False,1,Established & Improving
4,00000000000000000004,26.951888,184.807776,-0.031373,58.370362,3.466935,False,1,Established & Improving


## 1. Strategy assignment rules

**Cluster-level defaults** (the baseline recommendation for each health category):

- **Established & Improving** → P5 (Community-Based Monitoring) — stable, so the
  main need is sustained oversight, not intervention.
- **New but Catchment-Stressed** → P4 (Afforestation Around Water Bodies) — the
  water body itself is fine (it's new), but the surrounding land needs vegetation
  restoration.
- **Urban-Adjacent Growth Pressure** → P5 (Community-Based Monitoring) — see note
  below on why this is an imperfect fit.

**Per-water-body overrides** (checked individually, can stack on top of or replace
the cluster default):

- If **net water area change < 0%** (i.e. actually shrinking, despite whatever the
  cluster average says) → **P1 (Desilting) becomes the primary strategy**, overriding
  the cluster default. This is the clearest, most direct signal of a physically
  declining water body.
- If **catchment cropland > 50%** → add P6 (Sustainable Agricultural Practices) and
  P2 (Catchment Area Treatment) — heavy cropland right up to the water body's edge
  implicates farming practices and siltation risk specifically.
- If **catchment NDVI change < -0.02** (meaningful vegetation decline) → add P4
  (Afforestation), if not already assigned.
- If **catchment urban % > 10%** → add a P5 encroachment-watch flag.

**Honest limitation worth stating directly:** the existing six-strategy framework
doesn't have a dedicated "prevent urban encroachment" category — P5 (Community
Monitoring) is the closest fit for the Urban-Adjacent cluster, but it's an imperfect
match. This is worth naming explicitly in the report rather than silently forcing an
imperfect fit and hoping nobody asks.


In [2]:
def assign_strategies(row):
    strategies = []

    # Override: actual net water loss -> Desilting becomes PRIMARY regardless of cluster
    if row['water_pct_change_capped'] < 0:
        strategies.append('P1: Desilting of Shrinking Water Bodies')

    # Cluster-level default primary (only if no override fired above)
    if not strategies:
        if row['health_category'] == 'Established & Improving':
            strategies.append('P5: Community-Based Monitoring')
        elif row['health_category'] == 'New but Catchment-Stressed':
            strategies.append('P4: Afforestation Around Water Bodies')
        elif row['health_category'] == 'Urban-Adjacent Growth Pressure':
            strategies.append('P5: Community-Based Monitoring')

    # Secondary, stackable conditions
    if row['catchment_pct_cropland'] > 50:
        strategies.append('P6: Sustainable Agricultural Practices')
        strategies.append('P2: Catchment Area Treatment')

    if row['catchment_ndvi_change'] < -0.02 and 'P4: Afforestation Around Water Bodies' not in strategies:
        strategies.append('P4: Afforestation Around Water Bodies')

    if row['catchment_pct_urban'] > 10:
        strategies.append('P5: Community-Based Monitoring (encroachment watch)')

    return strategies

df['strategies'] = df.apply(assign_strategies, axis=1)
df['primary_strategy'] = df['strategies'].apply(lambda x: x[0])
df['num_strategies'] = df['strategies'].apply(len)

df[['water_body_id', 'health_category', 'primary_strategy', 'num_strategies']].head(10)


,water_body_id,health_category,primary_strategy,num_strategies
0,00000000000000000000,Established & Improving,P5: Community-Based Monitoring,2
1,00000000000000000001,New but Catchment-Stressed,P4: Afforestation Around Water Bodies,1
2,00000000000000000002,Established & Improving,P5: Community-Based Monitoring,4
3,00000000000000000003,Established & Improving,P5: Community-Based Monitoring,3
4,00000000000000000004,Established & Improving,P5: Community-Based Monitoring,4
5,00000000000000000005,Established & Improving,P1: Desilting of Shrinking Water Bodies,4
6,00000000000000000006,Urban-Adjacent Growth Pressure,P5: Community-Based Monitoring,5
7,00000000000000000007,Established & Improving,P5: Community-Based Monitoring,2
8,00000000000000000008,Urban-Adjacent Growth Pressure,P5: Community-Based Monitoring,4
9,00000000000000000009,New but Catchment-Stressed,P4: Afforestation Around Water Bodies,1


## 2. Check the results — does this look sensible?

In [3]:
print('Primary strategy distribution across all 92 water bodies:')
print(df['primary_strategy'].value_counts())
print()
print(f"Average number of stacked strategies per water body: {df['num_strategies'].mean():.2f}")
print(f"Max stacked strategies on a single water body: {df['num_strategies'].max()}")


Primary strategy distribution across all 92 water bodies:
primary_strategy
P5: Community-Based Monitoring             52
P4: Afforestation Around Water Bodies      32
P1: Desilting of Shrinking Water Bodies     8
Name: count, dtype: int64

Average number of stacked strategies per water body: 1.64
Max stacked strategies on a single water body: 5


**Reading this distribution:**

- **52 water bodies get P5 (Community-Based Monitoring) as primary** — the 51
  "Established & Improving" members that *don't* individually trigger the water-loss
  override, plus most of the Urban-Adjacent group.
- **32 get P4 (Afforestation) as primary** — essentially the entire "New but
  Catchment-Stressed" cluster, consistent with how that cluster was defined.
- **8 get P1 (Desilting) as primary** — these are exactly the individual override
  cases found earlier: water bodies quietly losing area despite sitting in the
  "positive average" cluster. These are arguably the single most actionable, highest
  -priority group in the entire health card, since desilting is a direct physical
  intervention, not a monitoring/prevention measure like the other two primaries.


## 3. Priority ranking for the report

Not every water body needs equal attention. A simple, defensible priority order:

1. **P1 primary (desilting)** — physically shrinking, most urgent
2. **High stacked-strategy count (3+)** — multiple compounding pressures at once
3. Everything else, grouped by health category


In [4]:
def priority_rank(row):
    if row['primary_strategy'].startswith('P1'):
        return 1
    elif row['num_strategies'] >= 3:
        return 2
    else:
        return 3

df['priority_rank'] = df.apply(priority_rank, axis=1)

priority_summary = df.groupby('priority_rank').agg(
    count=('water_body_id', 'count'),
    avg_area_ha=('area_ha_2024', 'mean')
).round(2)
priority_summary.index = ['1 - Urgent (shrinking)', '2 - Multiple pressures', '3 - Standard monitoring']
priority_summary


,count,avg_area_ha
1 - Urgent (shrinking),8,5.58
2 - Multiple pressures,18,33.41
3 - Standard monitoring,66,42.64


In [5]:
# The 8 highest-priority water bodies, ready to cross-reference against
# the field visit site selection
urgent = df[df['priority_rank'] == 1][
    ['water_body_id', 'area_ha_2024', 'water_pct_change_capped',
     'catchment_pct_cropland', 'catchment_pct_urban', 'strategies']
].sort_values('water_pct_change_capped')

urgent


,water_body_id,area_ha_2024,water_pct_change_capped,catchment_pct_cropland,catchment_pct_urban,strategies
19,00000000000000000013,0.570018,-0.353647,41.545159,0.000000,[P1: Desilting of Shrinking Water Bodies]
51,00000000000000000033,0.809118,-0.263064,0.000000,0.000000,[P1: Desilting of Shrinking Water Bodies]
52,00000000000000000034,5.790078,-0.165488,0.358893,0.272458,[P1: Desilting of Shrinking Water Bodies]
54,00000000000000000036,3.218829,-0.124306,4.032606,0.000000,[P1: Desilting of Shrinking Water Bodies]
67,00000000000000000043,0.635299,-0.119529,28.070942,0.105221,[P1: Desilting of Shrinking Water Bodies]
5,00000000000000000005,0.606128,-0.059481,72.871796,3.198579,"[P1: Desilting of Shrinking Water Bodies, P6: Sustainabl..."
68,00000000000000000044,4.354948,-0.050578,91.825827,2.914026,"[P1: Desilting of Shrinking Water Bodies, P6: Sustainabl..."
20,00000000000000000014,28.629473,-0.039972,88.257946,6.402568,"[P1: Desilting of Shrinking Water Bodies, P6: Sustainabl..."


**This `urgent` table is the direct answer to "which water bodies should we visit in
the field."** These 8 are shrinking on net, ranked worst-first — the strongest,
most defensible candidates for the mandatory field visit, since they're not just
statistically flagged but physically declining.


## 4. Export the final health card table

In [6]:
df['strategies_str'] = df['strategies'].apply(lambda x: '; '.join(x))

output_cols = ['water_body_id', 'area_ha_2024', 'water_pct_change_capped',
               'catchment_ndvi_change', 'catchment_pct_cropland', 'catchment_pct_urban',
               'is_new_since_2017', 'health_category', 'primary_strategy',
               'strategies_str', 'num_strategies', 'priority_rank']

final = df[output_cols].sort_values('priority_rank')
final.to_csv('../Sehore_WaterBody_Health_Card_FINAL.csv', index=False)

print('Saved: Sehore_WaterBody_Health_Card_FINAL.csv')
print(f'{len(final)} water bodies, {final["priority_rank"].eq(1).sum()} flagged urgent')
final.head(10)


Saved: Sehore_WaterBody_Health_Card_FINAL.csv
92 water bodies, 8 flagged urgent


,water_body_id,area_ha_2024,water_pct_change_capped,catchment_ndvi_change,catchment_pct_cropland,catchment_pct_urban,is_new_since_2017,health_category,primary_strategy,strategies_str,num_strategies,priority_rank
5,00000000000000000005,0.606128,-0.059481,-0.035636,72.871796,3.198579,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies; P6: Sustainable...,4,1
19,00000000000000000013,0.570018,-0.353647,0.011326,41.545159,0.000000,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies,1,1
20,00000000000000000014,28.629473,-0.039972,-0.001892,88.257946,6.402568,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies; P6: Sustainable...,3,1
52,00000000000000000034,5.790078,-0.165488,0.023521,0.358893,0.272458,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies,1,1
54,00000000000000000036,3.218829,-0.124306,0.011187,4.032606,0.000000,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies,1,1
51,00000000000000000033,0.809118,-0.263064,0.016021,0.000000,0.000000,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies,1,1
68,00000000000000000044,4.354948,-0.050578,0.048570,91.825827,2.914026,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies; P6: Sustainable...,3,1
67,00000000000000000043,0.635299,-0.119529,0.013683,28.070942,0.105221,False,Established & Improving,P1: Desilting of Shrinking Water Bodies,P1: Desilting of Shrinking Water Bodies,1,1
16,00000000000000000010,9.027167,2.011011,-0.008294,72.558623,3.579351,False,Established & Improving,P5: Community-Based Monitoring,P5: Community-Based Monitoring; P6: Sustainable Agricult...,3,2
24,00000000000000000018,80.977635,23.178524,-0.001224,80.041113,9.131246,False,Established & Improving,P5: Community-Based Monitoring,P5: Community-Based Monitoring; P6: Sustainable Agricult...,3,2


## Next step

`02_report_generation.ipynb` — takes this final CSV and auto-generates a per-water-body
(or per-priority-group) PDF health card, using the same Flask/ReportLab pattern as
Insight Engine: profile the data → generate narrative text per entry → assemble PDF.
